# Filling the Missing Square: Inpainting on AFHQ
The notebook only *drives* the project. The code lives in separate files:

| File | Purpose |
|---|---|
| `src/data/preprocessing.py`, `src/data/masks.py` | dataset, resize/normalise, train & fixed test masks |
| `src/eval/metrics.py`, `src/eval/evaluate.py` | PSNR / SSIM / LPIPS / FID / diversity, shared eval loop |
| `src/methods/mae.py` | Method 1 MAE |
| `src/methods/stylegan_inversion.py` | Method 2 |
| `src/methods/stylegan_noninversion.py` | Method 3 |
| `src/methods/mirror_conditioned.py` | Method 4 |
| `src/methods/repaint.py` | Method 5 |
| `src/methods/mirror_diffusion.py` | Method 6 |

Runs on Colab / Kaggle / local. On Colab choose a GPU runtime first.

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/ericacleo/dlpastia.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm
print(os.getcwd())

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print('NO GPU: Colab -> Runtime > Change runtime type > T4 GPU, then rerun')

## 1. Download data
Set `KAGGLE_API_TOKEN` (Colab: Secrets panel; local: env var). Never commit it. On a Kaggle notebook, add the dataset as an input and symlink it instead.

In [ ]:
# os.environ['KAGGLE_API_TOKEN'] = '...'   # or use Colab secrets
!python scripts/download_data.py
!python scripts/prepare_data.py

## 2. Data + fixed masks (`src/data/`)

In [ ]:
import torch, matplotlib.pyplot as plt
from src.data import get_datasets, load_config
cfg = load_config()
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))
fig, ax = plt.subplots(2, 6, figsize=(14,5))
for k in range(6):
    x, m, xm = test_ds[k*200]
    ax[0,k].imshow(((x+1)/2).permute(1,2,0)); ax[1,k].imshow(((xm+1)/2).permute(1,2,0))
    ax[0,k].axis('off'); ax[1,k].axis('off')
plt.show()

## 3. Evaluate (`src/eval/`)
Baselines first: they check the pipeline and give lower bounds. `MirrorPaste` shows what raw left-right symmetry is worth with no learning.

In [ ]:
from src.eval import evaluate
from src.methods import GreyFill, MirrorPaste
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
for mask in ['square', 'one_sided', 'centre']:
    for M in (GreyFill, MirrorPaste):
        print(mask, M.name, evaluate(M(), mask, n_images=100, device=dev))

## 4. Methods
Each method is a file in `src/methods/` implementing `fill(masked, mask, K)`. Fill in the file, then evaluate here.

### 4.1 MAE  -> `src/methods/mae.py`

In [ ]:
# from src.methods.mae import MAEInpainter

### 4.2 StyleGAN inversion  -> `src/methods/stylegan_inversion.py`

In [ ]:
# from src.methods.stylegan_inversion import StyleGANInversion

### 4.3 StyleGAN non-inversion  -> `src/methods/stylegan_noninversion.py`

In [ ]:
# from src.methods.stylegan_noninversion import CoModGANInpainter

### 4.4 Mirror-conditioned  -> `src/methods/mirror_conditioned.py`

In [ ]:
# from src.methods.mirror_conditioned import MirrorConditionedInpainter

### 4.5 RePaint  -> `src/methods/repaint.py`

In [ ]:
# from src.methods.repaint import RePaint

### 4.6 Mirror-guided RePaint (ours)  -> `src/methods/mirror_diffusion.py`
Decided: option (b), training-free adaptive mirror guidance on RePaint.

In [ ]:
# from src.methods.mirror_diffusion import MirrorGuidedRePaint

## 5. Results table + qualitative grids

In [ ]:
# collect evaluate(...) outputs for all methods (K=5 fills for stochastic ones)